<img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png"  width='15%'>  


<h1> Demo: LangChain Splitters</h1>
Developed by WeCloudData
<br></br>

In [2]:
!pip install langchain langchain-openai langchain-community langchain-chroma langchain-unstructured langchain-classic langgraph > /dev/null

In [3]:
# For the core building blocks
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.runnables import Runnable

# --- Text Splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter

# --- Retrievers
from langchain_classic.retrievers import ParentDocumentRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# --- Vector Stores
from langchain_chroma import Chroma
from langchain_community.vectorstores import Vectara

# --- Embeddings
from langchain_openai import OpenAIEmbeddings

# --- Storage
from langchain_core.stores import InMemoryStore

# --- Loaders
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import UnstructuredFileLoader

# ---- Retrieval
from langchain_classic.chains import RetrievalQA

# --- LLMs
from langchain_openai import ChatOpenAI

# --- Other Utilities
import urllib.request

# Setup

In [ ]:
#!wget https://s3.amazonaws.com/weclouddata/datasets/genai/rag/llama2.pdf

In [4]:
url = 'https://arxiv.org/pdf/2307.09288.pdf'
file_name = 'llama2.pdf'
urllib.request.urlretrieve(url, file_name)

('llama2.pdf', <http.client.HTTPMessage at 0x7e7df57dee40>)

In [32]:

def get_answer_character(doc_text, chunk_size: int, chunk_overlap: int, query: str):
    text_splitter = CharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs = text_splitter.split_documents(doc_text)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small",openai_api_key=OPENAI_API_KEY)
    vs = Chroma.from_documents(docs, embeddings)
    qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vs.as_retriever())
    return qa.invoke(query)

def get_answer_token(doc_text, chunk_size: int, chunk_overlap: int, query: str):
    """
    Splits documents by token and performs a RAG query.
    """
    text_splitter = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs = text_splitter.split_documents(doc_text)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small",openai_api_key=OPENAI_API_KEY)
    vs = Chroma.from_documents(docs, embeddings)
    qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vs.as_retriever())
    return qa.invoke(query)

def get_answer_recursive(doc_text, chunk_size: int, chunk_overlap: int, query: str):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs = text_splitter.split_documents(doc_text)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small",openai_api_key=OPENAI_API_KEY)
    vs = Chroma.from_documents(docs, embeddings)
    qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vs.as_retriever())
    return qa.invoke(query)

def get_answer_parent(doc_text, query: str):
    #from langchain.retrievers.document_compressors import EmbeddingsFilter

    llm = ChatOpenAI(
        model= "gpt-4o-mini",
        temperature=0,
        openai_api_key=OPENAI_API_KEY
    )
    # This text splitter is used to create the parent documents - The big chunks
    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=4000)
    # This text splitter is used to create the child documents - The small chunks
    # It should create documents smaller than the parent
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

    #docs = text_splitter.split_documents(doc_text)
    embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY
)

    vs = Chroma(collection_name="split_parents", embedding_function=embeddings)
    # The storage layer for the parent documents
    store = InMemoryStore()

    big_chunks_retriever = ParentDocumentRetriever(
        vectorstore=vs,
        docstore=store,
        child_splitter=child_splitter,
        parent_splitter=parent_splitter,
    )

    big_chunks_retriever.add_documents(doc_text)
    # compressor = LLMChainExtractor.from_llm(relevant_filter)
    # compression_retriever = ContextualCompressionRetriever(
    #     base_compressor=compressor, base_retriever=big_chunks_retriever,
    # )

    qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=big_chunks_retriever)
    return qa.invoke(query)
    # return vs.similarity_search(query)

In [33]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OpenAIAPIkey')

llm = ChatOpenAI(model_name = "gpt-4o-mini",
                 temperature=0,
                 openai_api_key=OPENAI_API_KEY)

In [34]:
%pip install --upgrade --quiet pypdf

In [40]:
from langchain_community.document_loaders import PyPDFLoader
#load llama
loader = PyPDFLoader("llama2.pdf")
doc_text = loader.load()
#split llama


In [41]:
query1 = "What is LLaMA 2 and what is it used for?"
query2 = "What are the main improvements in LLaMA 2 compared to LLaMA 1?"
query3 = "What datasets were used to train LLaMA"

In [37]:
#doc_text[33]

# Query 1:

### Q1: CharacterTextSplitter

In [42]:
print(type(doc_text))

<class 'list'>


In [43]:
query = query1

for chunk_size in [500, 1000, 2000]:
    for chunk_overlap in [0, 100]:
        response = get_answer_character(doc_text, chunk_size, chunk_overlap, query)
        print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n\n")

chunk=500, overlap=0, response={'query': 'What is LLaMA 2 and what is it used for?', 'result': 'LLaMA 2 is a family of pretrained and fine-tuned Large Language Models (LLMs) developed to enhance the capabilities of AI assistants. It includes models with scales ranging from 7 billion to 70 billion parameters. LLaMA 2 is designed to perform well on various tasks, including complex reasoning and expert knowledge applications, and is intended for use in chat interfaces, enabling interaction with humans.\n\nThe models have been fine-tuned to align with human preferences, improving their usability and safety. They are evaluated on benchmarks for helpfulness and safety, and they aim to provide competitive performance compared to existing open-source and some closed-source models. LLaMA 2 is part of ongoing efforts to advance AI alignment research and improve the safety of LLMs.'}


chunk=500, overlap=100, response={'query': 'What is LLaMA 2 and what is it used for?', 'result': 'LLaMA 2 is a f

### Q1: TokenTextSplitter

In [45]:
query = query1

for chunk_size in [500, 1000, 2000]:
    for chunk_overlap in [0, 100]:
        response = get_answer_token(doc_text, chunk_size, chunk_overlap, query)
        print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n\n")


chunk=500, overlap=0, response={'query': 'What is LLaMA 2 and what is it used for?', 'result': 'LLaMA 2 is a family of pretrained and fine-tuned large language models (LLMs) developed to enhance usability and safety in AI applications. It includes models with scales ranging from 7 billion to 70 billion parameters. LLaMA 2 and LLaMA 2-Chat are designed to perform competitively with existing open-source chat models and demonstrate competency comparable to some proprietary models on various evaluation benchmarks.\n\nLLaMA 2 is used for a range of applications, including but not limited to natural language understanding, text generation, code generation, and reading comprehension tasks. The models have been fine-tuned to align with human preferences, making them more helpful and safe for users. The development of LLaMA 2 also emphasizes transparency and reproducibility in AI research, allowing the community to build upon its methodologies and improve LLM safety.'}


chunk=500, overlap=100,

### Q1: RecrusiveTextSplitter

In [46]:
query = query1

for chunk_size in [500, 1000, 2000]:
    for chunk_overlap in [0, 100]:
        response = get_answer_recursive(doc_text, chunk_size, chunk_overlap, query)
        print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n")


chunk=500, overlap=0, response={'query': 'What is LLaMA 2 and what is it used for?', 'result': 'LLaMA 2 is a new technology, specifically a large language model (LLM) developed by Meta. It is designed to generate human-like text based on the input it receives. LLaMA 2 can be used for various applications, including chatbots, content generation, and other natural language processing tasks. However, it carries potential risks with use, and developers are advised to perform safety testing and tuning before deploying applications that utilize LLaMA 2.'}

chunk=500, overlap=100, response={'query': 'What is LLaMA 2 and what is it used for?', 'result': 'LLaMA 2 is a new technology, specifically a large language model (LLM) developed by Meta. It is designed for both research and commercial use, and it can generate text based on user prompts. LLaMA 2 is trained on a vast amount of data from publicly available sources and is intended to assist developers in creating applications that utilize nat

# Query 2:

### Q2: TokenTextSplitter

In [47]:
query = query2

for chunk_size in [500, 1000, 2000]:
    for chunk_overlap in [0, 100]:
        response = get_answer_token(doc_text, chunk_size, chunk_overlap, query)
        print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n\n")

chunk=500, overlap=0, response={'query': 'What are the main improvements in LLaMA 2 compared to LLaMA 1?', 'result': 'The main improvements in LLaMA 2 compared to LLaMA 1 include increased context length and the introduction of grouped-query attention (GQA).'}


chunk=500, overlap=100, response={'query': 'What are the main improvements in LLaMA 2 compared to LLaMA 1?', 'result': 'The main improvements in LLaMA 2 compared to LLaMA 1 include increased context length and the introduction of grouped-query attention (GQA).'}


chunk=1000, overlap=0, response={'query': 'What are the main improvements in LLaMA 2 compared to LLaMA 1?', 'result': 'The main improvements in LLaMA 2 compared to LLaMA 1 include increased context length and the introduction of grouped-query attention (GQA).'}


chunk=1000, overlap=100, response={'query': 'What are the main improvements in LLaMA 2 compared to LLaMA 1?', 'result': 'The main improvements in LLaMA 2 compared to LLaMA 1 include increased context length a

### Q2: RecrusiveTextSplitter

In [48]:
query = query2

for chunk_size in [500, 1000, 2000]:
    for chunk_overlap in [0, 200]:
        response = get_answer_recursive(doc_text, chunk_size, chunk_overlap, query)
        print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n")


chunk=500, overlap=0, response={'query': 'What are the main improvements in LLaMA 2 compared to LLaMA 1?', 'result': 'The main improvements in LLaMA 2 compared to LLaMA 1 include increased context length and the introduction of grouped-query attention (GQA).'}

chunk=500, overlap=200, response={'query': 'What are the main improvements in LLaMA 2 compared to LLaMA 1?', 'result': 'The main improvements in LLaMA 2 compared to LLaMA 1 include increased context length and the introduction of grouped-query attention (GQA).'}

chunk=1000, overlap=0, response={'query': 'What are the main improvements in LLaMA 2 compared to LLaMA 1?', 'result': 'The main improvements in LLaMA 2 compared to LLaMA 1 include increased context length and the introduction of grouped-query attention (GQA).'}

chunk=1000, overlap=200, response={'query': 'What are the main improvements in LLaMA 2 compared to LLaMA 1?', 'result': 'The main improvements in LLaMA 2 compared to LLaMA 1 include increased context length and 

### Q3: RecrusiveTextSplitter

In [50]:
query = query3

for chunk_size in [500, 1000, 2000]:
    for chunk_overlap in [0, 100]:
        response = get_answer_recursive(doc_text, chunk_size, chunk_overlap, query)
        print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n\n")

chunk=500, overlap=0, response={'query': 'What datasets were used to train LLaMA', 'result': "I don't know."}


chunk=500, overlap=100, response={'query': 'What datasets were used to train LLaMA', 'result': "I don't know."}


chunk=1000, overlap=0, response={'query': 'What datasets were used to train LLaMA', 'result': "I don't know."}


chunk=1000, overlap=100, response={'query': 'What datasets were used to train LLaMA', 'result': "I don't know."}


chunk=2000, overlap=0, response={'query': 'What datasets were used to train LLaMA', 'result': "I don't know."}


chunk=2000, overlap=100, response={'query': 'What datasets were used to train LLaMA', 'result': "I don't know."}




### Q3: Parent_splitter/Child splitter

In [52]:
query = query3

response = get_answer_parent(doc_text, query)
print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n\n")

chunk=2000, overlap=100, response={'query': 'What datasets were used to train LLaMA', 'result': "I don't know."}


